# PathSnap: Exploratory Data Analysis & Spatial Architecture Specification
**Course**: UE24CS352A Machine Learning | Department of CSE, PES University  
**Module**: Data Engineering & Trajectory Alignment  
**Project**: PathSnap – Distributed Map Matching & GPS Trajectory Reconstruction  

---

### Notebook Objectives:
1. **Raw Dataset Inspection**: Verify schema, formats, and physical integrity of `.track`, `.nodes`, `.arcs`, and `.route` files.
2. **Partition Inventory**: Validate partition distributions across 80 training trips, 10 student-testing trips, and 10 withheld evaluator trips.
3. **Temporal Analysis**: Quantify sampling rate fidelity (nominal 1 Hz), identify temporal anomalies, and characterize sensor dropouts.
4. **Spatial & Network Topology**: Ingest localized road networks, verify index invariants, analyze graph connectivity, and detect isolated nodes.
5. **Route Continuity & Alignment**: Investigate topological progression in `.route`, validate continuous node-to-node transitions, and benchmark unconstrained vs. monotonic GPS-to-route alignment.
6. **Candidate Arc Generation Strategy**: Formulate candidate search radii ($R \in [15, 100]	ext{m}$) and evaluate candidate recall curves to minimize candidate miss rates.
7. **Geospatial Folium Visualization**: Render interactive multi-layer maps displaying raw GPS coordinates, corridor road networks, and ground truth reference paths.


In [ ]:
import sys
from pathlib import Path

# Add repository root to path
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import folium

from src.data.loader import load_trip, load_track, load_nodes, load_arcs, load_route, build_network_graph
from src.data.inspector import scan_entire_dataset, inspect_single_trip
from src.data.alignment import (
    align_gps_to_route_monotonic,
    align_gps_to_route_unconstrained,
    evaluate_candidate_recall,
    project_point_to_segment
)
from src.visualization.eda_plots import (
    plot_sampling_rate_distribution,
    plot_trajectory_statistics,
    plot_network_topology,
    plot_candidate_recall_curve,
    plot_projection_error_distribution
)
from src.visualization.map_renderer import render_folium_trajectory_map

sns.set_theme(style="whitegrid")
print("Environment successfully initialized with PathSnap engineering modules.")

## 1. Dataset Partitions and Comprehensive Inventory
The raw dataset is organized into distinct directories for `train` and `student_test`.  
We load the compiled dataset inventory from `experiments/results/dataset_inventory.csv` to analyze global properties across all 90 accessible trips.

In [ ]:
inventory_path = repo_root / "experiments" / "results" / "dataset_inventory.csv"
df_inv = pd.read_csv(inventory_path, dtype={"trip_id": str})
df_inv["trip_id"] = df_inv["trip_id"].astype(str).str.zfill(8)

print(f"Total trips inventoried: {len(df_inv)}")
print(f"Partitions: Train = {(df_inv['partition'] == 'train').sum()} trips | Student Test = {(df_inv['partition'] == 'student_test').sum()} trips")

# Summary aggregates
summary_df = df_inv.groupby("partition").agg({
    "track_points": ["sum", "mean", "min", "max"],
    "trajectory_length_km": ["sum", "mean"],
    "duration_seconds": ["mean", lambda s: s.mean() / 60.0],
    "dt_1s_pct": "mean",
    "route_continuity_pct": "mean"
})
summary_df.columns = ["Total Points", "Avg Points", "Min Points", "Max Points", "Total km", "Avg km", "Avg Dur (s)", "Avg Dur (min)", "Avg 1Hz %", "Avg Continuity %"]
summary_df

## 2. Raw File Inspection & Coordinate Verification
Each trip directory contains four files:
- `.track`: whitespace-separated `<longitude> <latitude> <timestamp_seconds>`
- `.nodes`: whitespace-separated `<longitude> <latitude>` (row index = `node_id`)
- `.arcs`: whitespace-separated `<from_node_id> <to_node_id>` (row index = `arc_id`)
- `.route`: whitespace-separated `<arc_id>` (ground-truth sequence)

Let us inspect a canonical trip (`00000001`).

In [ ]:
sample_trip_id = "00000001"
trip = load_trip(repo_root / "data" / "raw" / "train" / sample_trip_id, partition="train")

print(f"Trip {sample_trip_id} Loaded:")
print(f"  - Track shape: {trip.track.shape} (N={trip.num_observations} GPS points)")
print(f"  - Nodes shape: {trip.nodes.shape} (V={trip.num_nodes} vertices)")
print(f"  - Arcs shape:  {trip.arcs.shape} (E={trip.num_arcs} directed edges)")
print(f"  - Route shape: {trip.route.shape if trip.route is not None else None} (M={trip.num_route_segments} route segments)")

print("\nSample Track Observations (first 3 rows):")
print(trip.track[:3])

print("\nSample Network Arcs (first 3 rows):")
print(trip.arcs[:3])

print("\nSample Route Traversed Arcs (first 5 rows):")
print(trip.route[:5])

## 3. Sampling Regularity and Timestamp Delta ($\Delta t$) Analysis
Nominally, GPS trackers sample observations at 1 Hz. We analyze the empirical time differences $\Delta t = 	au_{t} - 	au_{t-1}$ to identify sampling jitter, stopovers, and temporal sensor dropouts.

In [ ]:
timestamps = trip.track[:, 2]
dts = np.diff(timestamps)

pct_1s = np.isclose(dts, 1.0, atol=0.05).mean() * 100.0
print(f"Trip {sample_trip_id} Sampling Statistics:")
print(f"  - Total Duration: {timestamps[-1] - timestamps[0]:.1f} s ({((timestamps[-1] - timestamps[0])/60.0):.2f} min)")
print(f"  - Mean Delta t: {dts.mean():.4f} s | Median: {np.median(dts):.4f} s | Std: {dts.std():.4f} s")
print(f"  - Min Delta t:  {dts.min():.4f} s | Max Delta t: {dts.max():.4f} s")
print(f"  - Exactly 1 Hz: {pct_1s:.2f}%")

# Global 1Hz distribution across all trips
plt.figure(figsize=(10, 4))
sns.histplot(df_inv["dt_1s_pct"], bins=25, color="#2ca02c", edgecolor="black")
plt.title("Distribution of 1 Hz Compliance Rates across 90 Trips")
plt.xlabel("Percentage of Steps with Delta t ~ 1.0s (%)")
plt.ylabel("Trip Count")
plt.show()

## 4. Road Network Graph Topology & Connectivity
The road network is localized per trip. We construct a directed graph $G = (V, E)$ using NetworkX to verify:
1. Valid 0-based node and arc indices.
2. Degree distributions and isolated vertices.
3. Connected components (weakly and strongly connected).

In [ ]:
G = build_network_graph(trip.nodes, trip.arcs)
print(f"Graph Construction Successful:")
print(f"  - Number of Nodes: {G.number_of_nodes()}")
print(f"  - Number of Directed Edges: {G.number_of_edges()}")

in_degrees = [d for _, d in G.in_degree()]
out_degrees = [d for _, d in G.out_degree()]

print(f"Degree Metrics:")
print(f"  - Max In-Degree: {max(in_degrees)} | Max Out-Degree: {max(out_degrees)}")
print(f"  - Average Degree: {np.mean(in_degrees):.2f}")
print(f"  - Isolated Nodes (In=0, Out=0): {sum(1 for d1, d2 in zip(in_degrees, out_degrees) if d1 == 0 and d2 == 0)}")

## 5. Ground Truth Route Continuity & Observation-Level Alignment
A critical data discovery is that `.track` and `.route` are **not 1-to-1**:
- For Trip `00000001`, there are 1,315 GPS pings but 525 route segments.
- Consecutive arcs in `.route` form a topologically continuous chain: `arc[m].to_node == arc[m+1].from_node`.
- We benchmark **Unconstrained Spatial Projection** vs **Monotonic Forward Alignment** to assign reference road segments to individual GPS observations.

In [ ]:
# 1. Topological Continuity Verification
arcs = trip.arcs
route = trip.route
continuity_checks = [arcs[route[i], 1] == arcs[route[i+1], 0] for i in range(len(route)-1)]
continuity_rate = np.mean(continuity_checks) * 100.0
print(f"Route Topological Continuity: {continuity_rate:.2f}% ({sum(continuity_checks)} / {len(route)-1} continuous transitions)")

# 2. Alignment comparison
unconstrained_arcs, unconstrained_ridx, unconstrained_dists = align_gps_to_route_unconstrained(
    trip.track, trip.nodes, trip.arcs, trip.route
)
monotonic_arcs, monotonic_ridx, monotonic_dists = align_gps_to_route_monotonic(
    trip.track, trip.nodes, trip.arcs, trip.route
)

agreement_rate = (unconstrained_arcs == monotonic_arcs).mean() * 100.0
print(f"Alignment Agreement Rate: {agreement_rate:.2f}% of GPS observations agree exactly between unconstrained and monotonic alignment.")
print(f"Monotonic Alignment Orthogonal Distances: Median = {np.median(monotonic_dists):.2f}m | 95th Percentile = {np.percentile(monotonic_dists, 95):.2f}m")

## 6. Spatial Candidate Generation & Recall Evaluation
In map matching, a model ranks candidate segments within a spatial search radius $R$ of each GPS observation.
If the true road segment is outside radius $R$, the model suffers a **candidate miss**, placing an upper bound on route reconstruction accuracy.  
We evaluate candidate recall across multiple radii $R \in [15, 100]	ext{m}$.

In [ ]:
recalls = evaluate_candidate_recall(trip, search_radii=[15.0, 25.0, 35.0, 50.0, 75.0, 100.0])

print("Candidate Recall for Trip 00000001:")
for r, rec in recalls.items():
    print(f"  Radius {r:>5.1f}m -> Recall: {rec:>6.2f}% (Miss Rate: {100.0 - rec:>5.2f}%)")

# Recommendation: A candidate radius of 50m captures ~96% of true segments while keeping candidate counts small.

## 7. Interactive Geospatial Folium Map
We render the interactive Folium map for Trip `00000001`, visualizing the raw GPS points (blue), background road network corridor (gray), and the traversed reference route (green).

In [ ]:
map_output_path = repo_root / "reports" / "figures" / "trip_00000001_map.html"
m = render_folium_trajectory_map(trip, map_output_path, max_network_arcs=3000, network_buffer_m=500.0)
print(f"Folium Map saved to: {map_output_path}")
print("Map can be viewed in browser or rendered directly in Jupyter.")
# m  # Uncomment to render inline in Jupyter

## 8. Shared ML Data Contract & Architectural Decisions

Based on our empirical dataset exploration, the team establishes the following data contracts:

1. **Local Graph Scopes**: Road networks are localized per trip. Models must operate on **coordinate-invariant candidate features** (perpendicular distance, heading difference, velocity ratio, topological connectivity) rather than absolute coordinates or global arc IDs.
2. **Ground Truth Labeling**: Ground-truth candidate labels $y \in \{0, 1\}$ are assigned via **monotonic route projection**, matching each GPS ping $p_t$ to the active traversed segment in `.route`.
3. **Candidate Search Window**: We adopt a primary candidate radius $R = 50	ext{m}$, achieving $>95\%$ candidate recall while maintaining efficient inference latency.
4. **HMM Formulation**:
   - Hidden States: Spatial arc candidates $a_{t,k}$ within $R=50	ext{m}$.
   - Emission Probabilities: Calibrated classifier probabilities $P(y=1 \mid \mathbf{x})$.
   - Transition Probabilities: Shortest path network distance vs observed great-circle distance.
5. **Leakage Protection**: Strict trip-level partitioning (`GroupKFold` across 80 training trips) ensures zero cross-contamination. Evaluator-testing labels remain strictly server-side.